In [56]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/waynaali/company-docs/company_docs/it_policy.txt
/kaggle/input/datasets/waynaali/company-docs/company_docs/benefits.txt
/kaggle/input/datasets/waynaali/company-docs/company_docs/hr_policy.txt


In [57]:
pip install langchain langchain-openai pypdf 

Note: you may need to restart the kernel to use updated packages.


In [58]:
pip install langchain langchain-community langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [73]:
!pip install --upgrade google-generativeai

In [76]:
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

# Load secret
user_secrets = UserSecretsClient()
GOOGLE_API_KEY = user_secrets.get_secret("gemini_API_key")

# Configure Gemini
genai.configure(api_key=GOOGLE_API_KEY)

# Use the full explicit model path string
model = genai.GenerativeModel("models/gemini-2.5-flash")

# Test
response = model.generate_content("hello")
print(response.text)

Hello there! How can I help you today?


In [77]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv  # Load environment 
load_dotenv()  
# Load all text files from directory 
loader = DirectoryLoader(
    r"/kaggle/input/datasets/waynaali/company-docs/company_docs",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(documents[0].page_content[:200])

Loaded 3 documents
Information Technology Policy - IT Usage Guidelines

Password Security:
Employees must create strong passwords containing at least 12 characters including numbers, symbols, and uppercase letters.
Pass


In [78]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Characters per chunk
    chunk_overlap=50,      # Overlap between chunks
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")

print("\nSample chunks:")

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content)
    print(f"Length: {len(chunk.page_content)} chars")

Split into 5 chunks

Sample chunks:

Chunk 1:
Information Technology Policy - IT Usage Guidelines

Password Security:
Employees must create strong passwords containing at least 12 characters including numbers, symbols, and uppercase letters.
Passwords must be changed every 90 days.

Device Usage:
Company-issued laptops and devices should only be used for work-related tasks.
Installing unauthorized software is prohibited.
Length: 378 chars

Chunk 2:
Data Security:
Confidential company data must not be stored on personal devices.
Employees must use company-approved cloud storage for file sharing.

Email Usage:
Company email accounts should be used for official communication only.
Suspicious emails or phishing attempts must be reported to the IT department immediately.
Length: 323 chars

Chunk 3:
Employee Benefits - Company Benefits Policy

Health Insurance:
All full-time employees are eligible for company-sponsored health insurance after 30 days of employment.
The company covers 70% of t

In [79]:
def simple_search(query, chunks, top_k=3):    
    """     
    Simple keyword-based search     
    Returns top_k most relevant chunks     
    """     
    query_lower = query.lower()          
# Score each chunk     
    scored_chunks = []     
    for chunk in chunks:         
        content_lower = chunk.page_content.lower()                  
# Count keyword matches         
        score = 0         
        for word in query_lower.split():             
            score += content_lower.count(word)                  
            if score > 0:             
                scored_chunks.append((score, chunk))         
# Sort by score and return top k     
                scored_chunks.sort(reverse=True, key=lambda x: x[0])     
                return [chunk for score, chunk in scored_chunks[:top_k]]  
# Test it 
                query = 'What is the vacation policy?' 
                relevant = simple_search(query, chunks)  
                print(f'Found {len(relevant)} relevant chunks:') 
                for i, chunk in enumerate(relevant):     
                    print(f'\n--- Chunk {i+1} ---')     
                    print(chunk.page_content) 

In [80]:
# Test multiple queries 
test_queries = [     
    'How many vacation days do employees get?',     
    'What is the remote work policy?',     
    'Tell me about parental leave', 
]  
for query in test_queries:     
    print(f'\nQuery: {query}')     
    results = simple_search(query, chunks, top_k=2)     
    print(f'Found {len(results)} relevant chunks')    
    if results:
        print(f'Top result: {results[0].page_content[:100]}...') 


Query: How many vacation days do employees get?
Found 1 relevant chunks
Top result: Information Technology Policy - IT Usage Guidelines

Password Security:
Employees must create strong...

Query: What is the remote work policy?
Found 1 relevant chunks
Top result: Information Technology Policy - IT Usage Guidelines

Password Security:
Employees must create strong...

Query: Tell me about parental leave
Found 1 relevant chunks
Top result: Data Security:
Confidential company data must not be stored on personal devices.
Employees must use ...


In [83]:
# Install Gemini integration
!pip install -q langchain-google-genai

import os
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API key from Kaggle Secrets
user_secrets = UserSecretsClient()

# FIX: Changed "gemini_API_key" to "GOOGLE_API_KEY" so LangChain can detect it automatically
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("gemini_api_key")

# Initialize Gemini model (Using the standard 1.5-flash)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

def rag_query(query, chunks, top_k=3):
    # Retrieve relevant chunks
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return "No relevant information found in documents."

    # Combine retrieved chunks into context
    context = "\n\n---\n\n".join(
        [chunk.page_content for chunk in relevant_chunks]
    )

    # Build prompt
    prompt = f"""
You are a helpful HR assistant.

Answer ONLY using the context below.

If the answer is not present in the context, say:
"Not found in documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # Query Gemini
    response = llm.invoke(prompt)

    return response.content

In [84]:
# Test questions
questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the parental leave policy?',
    'What is the dress code?'  # Not in docs
]

for question in questions:
    print(f'\n{"=" * 60}')
    print(f'Q: {question}')
    print(f'{"=" * 60}')

    answer = rag_query(question, chunks)
    print(f'A: {answer}')


Q: How many vacation days do full-time employees get?
A: Not found in documents.

Q: Can employees work from home?
A: Not found in documents.

Q: What is the parental leave policy?
A: Not found in documents.

Q: What is the dress code?
A: Not found in documents.


In [85]:
# BONUS: Compare with vs without RAG

def ask_without_rag(question):
    """
    Ask GPT directly (no context)
    """
    messages = [
        {'role': 'system', 'content': 'You are a helpful HR assistant.'},
        {'role': 'user', 'content': question}
    ]

    response = llm.invoke(messages)
    return response.content


# Compare
question = 'How many vacation days do employees get?'

print('WITHOUT RAG:')
print(ask_without_rag(question))

print('\nWITH RAG:')
print(rag_query(question, chunks))

WITHOUT RAG:
That's a great question, and the answer can vary quite a bit!

The number of vacation days an employee receives depends entirely on the **company's specific policy**.

Here are the common factors that influence vacation day accrual:

1.  **Company Policy:** This is the biggest factor. Every company sets its own vacation/Paid Time Off (PTO) policy.
2.  **Employee Tenure:** Many companies offer more vacation days as an employee gains more years of service. For example, you might start with 10 days, move to 15 after 3 years, and 20 after 5 years.
3.  **Full-time vs. Part-time Status:** Part-time employees often accrue vacation days on a pro-rata basis (e.g., if full-time gets 10 days, a half-time employee might get 5).
4.  **Role or Level:** In some organizations, senior roles or executives might have different vacation benefits.
5.  **Combined PTO vs. Separate Vacation/Sick:** Some companies have a single "PTO" bank that covers all time off (vacation, sick, personal), while 

In [86]:
def ask_without_rag(question):
    """
    Ask GPT directly (no context)
    """
    messages = [
        {'role': 'system', 'content': 'You are a helpful HR assistant.'},
        {'role': 'user', 'content': question}
    ]

    response = llm.invoke(messages)
    return response.content


# Compare
question = 'How many vacation days do employees get?'

print('WITHOUT RAG:')
print(ask_without_rag(question))

print('\nWITH RAG:')
print(rag_query(question, chunks))

WITHOUT RAG:
That's a great question, and the answer can vary quite a bit!

The number of vacation days an employee receives depends entirely on the **company's specific policy**.

Here are the common factors that influence vacation day accrual:

1.  **Company Policy:** This is the biggest factor. Every company sets its own vacation/Paid Time Off (PTO) policy.
2.  **Employee Tenure:** Many companies offer more vacation days as an employee gains more years of service. For example, you might start with 10 days, move to 15 after 3 years, and 20 after 5 years.
3.  **Full-time vs. Part-time Status:** Part-time employees often accrue vacation days on a pro-rata basis (e.g., if full-time gets 10 days, a half-time employee might get 5).
4.  **Role or Level:** In some organizations, senior roles or executives might have different vacation benefits.
5.  **Combined PTO vs. Separate Vacation/Sick:** Some companies have a single "PTO" bank that covers all time off (vacation, sick, personal), while 